# Config Combo Comparison
Aggregates metrics across **all documents and both domains**, then shows mean scores for each of the 16 coreference-chunking config combinations.

**Configuration dimensions (all binary):**
- `method`: `most_recent` vs `most_recent_low_accessible`
- `sent_incl`: `full_sentence_included` vs `full_sentence_excluded`
- `weighted`: `weighted` vs `unweighted`
- `strategy`: `conservative` vs `liberal`

**Metrics (direction):**
- `cluster_break_rate` — lower is better
- `edge_cut_rate` — lower is better
- `entity_concentration` — higher is better

In [9]:
import json
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
from metrics import cluster_break_rate, edge_cut_rate, entity_concentration

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

METRICS = ['cluster_break_rate', 'edge_cut_rate', 'entity_concentration']
CONFIG_DIMS = ['method', 'sent_incl', 'weighted', 'strategy']

# Documents per domain (must have a matching clusters/ file)
DOCS = {
    'financial': [
        '2019-avni123118form10k.txt',
        '2020-avni123119form10k.txt',
        '2020-f10k2019_boxscorebrands.txt',
        '2020-form10-k.txt',
        '2021-avni123120form10k.txt',
        '2021-f10k2020_boxscorebrands.txt',
        '2021-form10-k.txt',
        '2022-aqb-20211231x10k.txt',
        '2022-avni123121form10k.txt',
        '2022-f10k2021_boxscorebrands.txt',
    ],
    'paper': [
        '1612.04662.md',
        '1709.03082.md',
        '1802.03426.md',
        '1802.08129.md',
        '1803.08375.md',
        '2405.20774.md',
        '2405.20974.md',
        '2405.21018.md',
        '2405.21040.md',
        '2405.21046.md',
    ],
}

In [10]:
def mention_char_to_chunk(mention_start, boundaries):
    for i, boundary in enumerate(boundaries[1:], 1):
        if mention_start < boundary:
            return i - 1
    return len(boundaries) - 2


def build_mention_chunks(chains, chunk_texts):
    boundaries = [0]
    for text in chunk_texts:
        boundaries.append(boundaries[-1] + len(text))
    mention_chunks = [
        [mention_char_to_chunk(m[0], boundaries) for m in chain]
        for chain in chains
    ]
    return mention_chunks, len(chunk_texts)


def collect_all_records():
    """Return a DataFrame with one row per (domain, doc, config combo)."""
    records = []
    for domain, files in DOCS.items():
        for source_file in files:
            source_stem = Path(source_file).stem
            cluster_path = Path('clusters') / domain / f'{source_stem}.json'
            if not cluster_path.exists():
                print(f'WARNING: missing cluster file {cluster_path}, skipping')
                continue
            with open(cluster_path) as f:
                chains = json.load(f)

            chunk_dir = Path('chunks') / domain / source_file
            for chunks_path in sorted(chunk_dir.rglob('chunks_output.json')):
                parts = chunks_path.parts  # [..., domain, source_file, method, sent_incl, weighted, strategy, filename]
                with open(chunks_path) as f:
                    data = json.load(f)
                mention_chunks, total_chunks = build_mention_chunks(chains, data['chunks'])
                records.append({
                    'domain':      domain,
                    'doc':         source_file,
                    'method':      parts[3],
                    'sent_incl':   parts[4],
                    'weighted':    parts[5],
                    'strategy':    parts[6],
                    'n_chunks':    data['metadata']['total_chunks_generated'],
                    'cluster_break_rate':   cluster_break_rate(mention_chunks),
                    'edge_cut_rate':        edge_cut_rate(mention_chunks),
                    'entity_concentration': entity_concentration(mention_chunks, total_chunks),
                })
    return pd.DataFrame(records)


def collect_baseline_records(baseline_dir):
    """Return a DataFrame with one row per doc for a given baseline folder."""
    records = []
    baseline_path = Path(baseline_dir)
    for domain, files in DOCS.items():
        for source_file in files:
            source_stem = Path(source_file).stem
            cluster_path = Path('clusters') / domain / f'{source_stem}.json'
            chunks_path  = baseline_path / domain / source_file / 'chunks_output.json'
            if not cluster_path.exists() or not chunks_path.exists():
                continue
            with open(cluster_path) as f:
                chains = json.load(f)
            with open(chunks_path) as f:
                data = json.load(f)
            mention_chunks, total_chunks = build_mention_chunks(chains, data['chunks'])
            records.append({
                'cluster_break_rate':   cluster_break_rate(mention_chunks),
                'edge_cut_rate':        edge_cut_rate(mention_chunks),
                'entity_concentration': entity_concentration(mention_chunks, total_chunks),
            })
    return pd.DataFrame(records)


# Discover all baseline dirs automatically
BASELINE_DIRS = sorted(p for p in Path('.').iterdir() if p.is_dir() and p.name.startswith('chunks_baseline'))
print(f'Found baseline dirs: {[str(p) for p in BASELINE_DIRS]}')

print('Collecting coreference records...')
raw_df = collect_all_records()
print(f'Done. {len(raw_df)} rows ({raw_df["doc"].nunique()} docs x up to 16 configs)')


Found baseline dirs: ['chunks_baseline_100_tok', 'chunks_baseline_300_tok']
Done. 320 rows (20 docs x up to 16 configs)


## Mean metrics per config combo — all documents

In [11]:
combo_df = (
    raw_df
    .groupby(CONFIG_DIMS)[METRICS]
    .mean()
    .reset_index()
    .sort_values('cluster_break_rate')
    .reset_index(drop=True)
)

# Build one reference row per baseline dir
baseline_rows = []
for bdir in BASELINE_DIRS:
    braw = collect_baseline_records(bdir)
    if braw.empty:
        print(f'No data for {bdir.name}, skipping.')
        continue
    baseline_rows.append({
        'method':    bdir.name,
        'sent_incl': '—',
        'weighted':  '—',
        'strategy':  '—',
        **braw[METRICS].mean().to_dict(),
    })

if baseline_rows:
    baselines_df = pd.DataFrame(baseline_rows)
    display_df = pd.concat([baselines_df, combo_df], ignore_index=True)
else:
    print('No baseline data found (run generate_baseline_chunks.py first).')
    display_df = combo_df

display_df[METRICS] = display_df[METRICS].astype(float).round(4)
display(display_df)


,method,sent_incl,weighted,strategy,cluster_break_rate,edge_cut_rate,entity_concentration
0,chunks_baseline_100_tok,—,—,—,0.6495,0.7957,0.4444
1,chunks_baseline_300_tok,—,—,—,0.4308,0.6320,0.4124
2,most_recent,full_sentence_excluded,unweighted,conservative,0.2634,0.5164,0.4011
3,most_recent,full_sentence_excluded,weighted,conservative,0.2636,0.5184,0.3968
4,most_recent,full_sentence_included,unweighted,conservative,0.2980,0.5300,0.3882
5,most_recent,full_sentence_included,weighted,conservative,0.3046,0.5348,0.3871
6,most_recent_low_accessible,full_sentence_included,unweighted,conservative,0.3233,0.5417,0.3799
7,most_recent_low_accessible,full_sentence_included,weighted,conservative,0.3279,0.5415,0.3785
8,most_recent,full_sentence_excluded,unweighted,liberal,0.3306,0.5666,0.4564
9,most_recent,full_sentence_excluded,weighted,liberal,0.3317,0.5689,0.4575


## Same table, broken out by domain

In [12]:
for domain in raw_df['domain'].unique():
    domain_df = (
        raw_df[raw_df['domain'] == domain]
        .groupby(CONFIG_DIMS)[METRICS]
        .mean()
        .reset_index()
        .sort_values('cluster_break_rate')
        .reset_index(drop=True)
        .round(4)
    )
    print(f'\n=== {domain.upper()} ===')
    display(domain_df)


=== FINANCIAL ===


,method,sent_incl,weighted,strategy,cluster_break_rate,edge_cut_rate,entity_concentration
0,most_recent,full_sentence_excluded,unweighted,conservative,0.2678,0.5379,0.3904
1,most_recent,full_sentence_excluded,weighted,conservative,0.2691,0.5409,0.3849
2,most_recent,full_sentence_included,unweighted,conservative,0.2926,0.5455,0.3828
3,most_recent,full_sentence_included,weighted,conservative,0.2976,0.5469,0.3812
4,most_recent_low_accessible,full_sentence_excluded,unweighted,conservative,0.3169,0.5566,0.3784
5,most_recent_low_accessible,full_sentence_excluded,weighted,conservative,0.3231,0.5570,0.3775
6,most_recent_low_accessible,full_sentence_included,unweighted,conservative,0.3258,0.5519,0.3729
7,most_recent_low_accessible,full_sentence_included,weighted,conservative,0.3380,0.5532,0.3707
8,most_recent,full_sentence_excluded,weighted,liberal,0.3387,0.5895,0.4436
9,most_recent,full_sentence_excluded,unweighted,liberal,0.3423,0.5913,0.4460



=== PAPER ===


,method,sent_incl,weighted,strategy,cluster_break_rate,edge_cut_rate,entity_concentration
0,most_recent,full_sentence_excluded,weighted,conservative,0.2581,0.4959,0.4088
1,most_recent,full_sentence_excluded,unweighted,conservative,0.2590,0.4949,0.4118
2,most_recent,full_sentence_included,unweighted,conservative,0.3034,0.5146,0.3935
3,most_recent,full_sentence_included,weighted,conservative,0.3116,0.5227,0.3931
4,most_recent_low_accessible,full_sentence_included,weighted,conservative,0.3178,0.5299,0.3863
5,most_recent,full_sentence_excluded,unweighted,liberal,0.3189,0.5420,0.4669
6,most_recent_low_accessible,full_sentence_included,unweighted,conservative,0.3207,0.5315,0.3869
7,most_recent,full_sentence_excluded,weighted,liberal,0.3248,0.5482,0.4714
8,most_recent,full_sentence_included,unweighted,liberal,0.3434,0.5502,0.4547
9,most_recent,full_sentence_included,weighted,liberal,0.3510,0.5560,0.4519


## Best config per metric
Quick reference: which combo wins on each metric individually.

In [13]:
print('Best cluster_break_rate (lowest):')
display(combo_df.nsmallest(3, 'cluster_break_rate')[CONFIG_DIMS + METRICS])

print('\nBest edge_cut_rate (lowest):')
display(combo_df.nsmallest(3, 'edge_cut_rate')[CONFIG_DIMS + METRICS])

print('\nBest entity_concentration (highest):')
display(combo_df.nlargest(3, 'entity_concentration')[CONFIG_DIMS + METRICS])

Best cluster_break_rate (lowest):


,method,sent_incl,weighted,strategy,cluster_break_rate,edge_cut_rate,entity_concentration
0,most_recent,full_sentence_excluded,unweighted,conservative,0.2634,0.5164,0.4011
1,most_recent,full_sentence_excluded,weighted,conservative,0.2636,0.5184,0.3968
2,most_recent,full_sentence_included,unweighted,conservative,0.2980,0.5300,0.3882



Best edge_cut_rate (lowest):


,method,sent_incl,weighted,strategy,cluster_break_rate,edge_cut_rate,entity_concentration
0,most_recent,full_sentence_excluded,unweighted,conservative,0.2634,0.5164,0.4011
1,most_recent,full_sentence_excluded,weighted,conservative,0.2636,0.5184,0.3968
2,most_recent,full_sentence_included,unweighted,conservative,0.2980,0.5300,0.3882



Best entity_concentration (highest):


,method,sent_incl,weighted,strategy,cluster_break_rate,edge_cut_rate,entity_concentration
7,most_recent,full_sentence_excluded,weighted,liberal,0.3317,0.5689,0.4575
6,most_recent,full_sentence_excluded,unweighted,liberal,0.3306,0.5666,0.4564
10,most_recent,full_sentence_included,unweighted,liberal,0.3480,0.5721,0.4426
